# 06 — Fusion multimodale de bout en bout
Parcelle de démo → observation → pipeline complet → rapport JSON validé.

In [ ]:
# --- Bootstrap (Kaggle or local) -------------------------------------------------
# Kaggle: Settings > Internet = ON ; Add-ons > Secrets > add "MODEL_API_KEY" (and "GITHUB_TOKEN" if the repo is private)
import os, subprocess, sys
REPO = "code-htech07/sina-meta"
BRANCH = "ccr-c8fab4b3-0jmvi1"  # change to "main" once merged
if os.path.exists("engine"):            # already inside the repo (local run)
    pass
else:
    if not os.path.exists("sina-meta"):
        url = f"https://github.com/{REPO}.git"
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
            url = f"https://{tok}@github.com/{REPO}.git"
        except Exception:
            pass
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "sina-meta"], check=True,
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)  # silent: never print a tokenised URL
    os.chdir("sina-meta")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, os.getcwd())
from engine.config import describe_secret_status
from engine.logging_utils import setup_logging
setup_logging("WARNING")
print(describe_secret_status())  # shows only presence/length of the key, never its value

In [ ]:
!python scripts/seed_demo.py --analyze

In [ ]:
from engine.services import MaanaTerraService
rep = MaanaTerraService().list_analyses(limit=1)[0]
print(rep.status, rep.caution_level, rep.caution_rationale)
for h in rep.primary_hypotheses: print(h.problem_id, h.status, h.support_level, h.weather_consistency)
for c in rep.contradictions: print('CONTRADICTION', c.origin, c.description)
print(rep.model_dump_json(indent=1)[:4000])